# Generador de visuales para el README — FutBot MX

**Para qué sirve:** producir **todos los elementos visuales** del README (GIFs cortos,
PNGs y gráficas) a partir de los artefactos ya generados por el pipeline. Las imágenes y
GIFs se guardan **versionados** en `assets/readme/` (subcarpetas `gifs/` y `png/`); los
mp4 intermedios viven en `outputs/` (git-ignored).

**Cómo leerlo:**
- Celdas **CPU** (gifs desde mp4 existentes, métricas/gráficas desde JSON): corren tal cual.
- Celdas **`RUN_HEAVY`** (cargan SAM3): solo en el **pod (GPU)** — segmentación en vivo.
- Cada celda **salta** si le faltan insumos (p. ej. los clips superior nuevos cuyo
  tracking se corre en el pod con `00_prepare_clips.py`).

**Vistas:** el pipeline base (detección/segmentación/tracking) aplica a **cualquier** clip;
homografía/eventos/broadcast/heatmap solo a **cámara superior**.

## 0 — Setup, rutas y helper `mp4_to_gif`

In [ ]:
import json
import os
from pathlib import Path

from src.utils import PROJECT_ROOT


def _envflag(name: str, default: bool = False) -> bool:
    """Lee un flag booleano de variable de entorno (1/true/yes/on/si)."""
    v = os.environ.get(name)
    return default if v is None else v.strip().lower() in ("1", "true", "yes", "on", "si", "sí")


# Flags: por defecto OFF (CPU local). En el pod, enciéndelos por entorno sin editar el notebook:
#   RUN_HEAVY=1 jupyter nbconvert --to notebook --execute --inplace <este_notebook>
RUN_HEAVY = _envflag("RUN_HEAVY")          # True SOLO en el pod (GPU): segmentación SAM3 en vivo
RUN_BROADCAST = _envflag("RUN_BROADCAST")  # True para RE-renderizar el broadcast (si no, reusa el mp4)

ASSETS = PROJECT_ROOT / "assets" / "readme"
GIFS, PNGS = ASSETS / "gifs", ASSETS / "png"
for d in (GIFS, PNGS):
    d.mkdir(parents=True, exist_ok=True)
print("salida versionada ->", ASSETS)
print(f"RUN_HEAVY={RUN_HEAVY}  RUN_BROADCAST={RUN_BROADCAST}")


def mp4_to_gif(src, dst, *, start=0.0, dur=5.0, fps=12, scale=0.5):
    """Convierte una ventana [start, start+dur) de un mp4 a GIF (downscaled, loop)."""
    import cv2
    import imageio.v2 as iio

    src, dst = Path(src), Path(dst)
    if not src.exists():
        print("  (skip gif) no existe:", src)
        return None
    rdr = iio.get_reader(str(src))
    src_fps = rdr.get_meta_data().get("fps", 30) or 30
    i0, i1 = int(start * src_fps), int((start + dur) * src_fps)
    step = max(1, round(src_fps / fps))
    frames = []
    for i, fr in enumerate(rdr):
        if i < i0:
            continue
        if i >= i1:
            break
        if (i - i0) % step:
            continue
        if scale != 1:
            h, w = fr.shape[:2]
            fr = cv2.resize(fr, (int(w * scale), int(h * scale)))
        frames.append(fr)
    rdr.close()
    if not frames:
        print("  (skip gif) sin frames en la ventana:", src)
        return None
    iio.mimsave(str(dst), frames, fps=fps, loop=0)
    print(f"  GIF -> {dst}  ({len(frames)} frames)")
    return dst

## 1 — Registro de fuentes (clip → tracking JSON + clip crudo)

In [2]:
def _first(*cands):
    for c in cands:
        if c and Path(c).exists():
            return Path(c)
    return None

OUT = PROJECT_ROOT / "outputs"

# Genéricos (<1 min, NO superior): pipeline base. JSON de tracking del benchmark.
GENERICOS = {}
for stem, raw in [
    ("video-714_singular_display", "data/raw/17Abril/video-714_singular_display.mov"),
    ("video-597_singular_display", "data/raw/17Abril/video-597_singular_display.mov"),
]:
    tj = _first(
        OUT / f"inference/trackers/yolo_sam3+bytetrack/{stem}/{stem}.json",
        OUT / f"inference/trackers/sam3_text+bytetrack/{stem}/{stem}.json",
    )
    GENERICOS[stem] = {"raw": _first(PROJECT_ROOT / raw), "tracking_json": tj}

# Cámara superior: post-proceso completo (homografía/eventos/broadcast/heatmap).
SUPERIOR = {}
for stem, raw in [
    ("IMG_9933_5m30", OUT / "eventos/IMG_9933_5m30/IMG_9933_5m30.mp4"),
    ("IMG_9933_8m00", OUT / "fase5_clips/IMG_9933_8m00.mp4"),
    ("IMG_9938_5m00", OUT / "fase5_clips/IMG_9938_5m00.mp4"),
]:
    tj = _first(OUT / f"inference/fase5_clips/{stem}/{stem}.json")
    SUPERIOR[stem] = {"raw": _first(raw), "tracking_json": tj}

print("GENÉRICOS:")
for k, v in GENERICOS.items():
    print(f"  {k}: json={'ok' if v['tracking_json'] else 'FALTA'}  raw={'ok' if v['raw'] else 'FALTA'}")
print("SUPERIOR:")
for k, v in SUPERIOR.items():
    print(f"  {k}: json={'ok' if v['tracking_json'] else 'FALTA(pod)'}  raw={'ok' if v['raw'] else 'FALTA'}")

GENÉRICOS:
  video-714_singular_display: json=ok  raw=ok
  video-597_singular_display: json=ok  raw=ok
SUPERIOR:
  IMG_9933_5m30: json=ok  raw=ok
  IMG_9933_8m00: json=FALTA(pod)  raw=ok
  IMG_9938_5m00: json=FALTA(pod)  raw=ok


## 2 — Pipeline base: overlay de tracking (`obj_id`)
GIF con caja + `nombre #id` + estela por `obj_id`. Aplica a **cualquier** clip; aquí los
genéricos. CPU (reusa el tracking JSON + el clip crudo).

In [3]:
from src.core.track_overlay import render_obj_id_overlay

for stem, src in GENERICOS.items():
    if not (src["tracking_json"] and src["raw"]):
        print(f"[{stem}] faltan insumos -> skip"); continue
    print(f"[{stem}] overlay obj_id…")
    mp4 = render_obj_id_overlay(src["tracking_json"], video_path=src["raw"], trajectory_window=30)
    mp4_to_gif(mp4, GIFS / f"tracking_{stem}.gif", start=0.0, dur=4.0, fps=10, scale=0.32)

[video-714_singular_display] overlay obj_id…


  GIF -> /mnt/datos/code/ai/futbot/assets/readme/gifs/tracking_video-714_singular_display.gif  (40 frames)
[video-597_singular_display] overlay obj_id…


  GIF -> /mnt/datos/code/ai/futbot/assets/readme/gifs/tracking_video-597_singular_display.gif  (40 frames)


## 3 — Pipeline base: segmentación (SAM3) — `RUN_HEAVY`
Still PNG de la segmentación en un frame (color por clase). Requiere SAM3 (GPU).

In [4]:
if RUN_HEAVY:
    import cv2
    from src.core.sam3_loader import load_sam3
    from src.core.segmentation import detect_classes_in_frame
    from src.core.overlay import overlay_detections
    from src.core.frame_extraction import extract_frames

    bundle = load_sam3()
    stem, src = next(iter(GENERICOS.items()))
    frame = extract_frames(src["raw"])[10]
    dets = detect_classes_in_frame(frame, classes=None, bundle=bundle)
    comp = overlay_detections(frame, dets)
    out = PNGS / f"segmentacion_{stem}.png"
    cv2.imwrite(str(out), cv2.cvtColor(comp, cv2.COLOR_RGB2BGR))
    print("PNG segmentación ->", out)
else:
    print("[RUN_HEAVY=False] segmentación SAM3 omitida (córrela en el pod).")

[RUN_HEAVY=False] segmentación SAM3 omitida (córrela en el pod).


## 4 — Cámara superior: broadcast (el entregable)
GIF + PNG de muestra del video de espectador (marcador, banner de gol, posesión, eventos,
minimap cenital, heatmap, homografía embebida). Reusa el mp4 si existe; con
`RUN_BROADCAST=True` lo re-renderiza. CPU.

In [5]:
from src.core.event_broadcast_overlay import render_broadcast_overlay
from src.core.events_schema import events_paths
import shutil

for stem, src in SUPERIOR.items():
    if not (src["tracking_json"] and src["raw"]):
        print(f"[{stem}] sin tracking JSON (pod) -> skip"); continue
    bmp4 = events_paths(stem, "broadcast", "mp4")
    bpng = events_paths(stem, "broadcast", "png")
    if RUN_BROADCAST or not bmp4.exists():
        print(f"[{stem}] render broadcast…")
        res = render_broadcast_overlay(src["tracking_json"], clip=src["raw"],
                                       layout=2, goal_source="strict", use_kalman=True,
                                       progress=True)
        bmp4, bpng = Path(res.video), (Path(res.sample_png) if res.sample_png else bpng)
    mp4_to_gif(bmp4, GIFS / f"broadcast_{stem}.gif", start=2.0, dur=6.0, fps=12, scale=0.4)
    if bpng and Path(bpng).exists():
        shutil.copy(bpng, PNGS / f"broadcast_{stem}.png")
        print("  PNG muestra ->", PNGS / f"broadcast_{stem}.png")

  GIF -> /mnt/datos/code/ai/futbot/assets/readme/gifs/broadcast_IMG_9933_5m30.gif  (90 frames)
  PNG muestra -> /mnt/datos/code/ai/futbot/assets/readme/png/broadcast_IMG_9933_5m30.png
[IMG_9933_8m00] sin tracking JSON (pod) -> skip
[IMG_9938_5m00] sin tracking JSON (pod) -> skip


## 5 — Cámara superior: heatmap de ocupación + zonas
Mapas en cm reales desde la métrica (homografía por líneas). CPU.

In [6]:
from src.core.metric_positions import compute_metric_positions
from src.core.metric_heatmap import compute_heatmaps, write_heatmap_png
from src.core.metric_field_zones import compute_field_zones, write_zones_png

for stem, src in SUPERIOR.items():
    if not (src["tracking_json"] and src["raw"]):
        print(f"[{stem}] sin insumos -> skip"); continue
    print(f"[{stem}] métrica (homografía por líneas)…")
    metric = compute_metric_positions(src["tracking_json"], video=src["raw"], homography="lines")
    if not any(p.xy_cm is not None for p in metric.posiciones):
        print(f"[{stem}] homografía degradada -> skip heatmap/zonas"); continue
    heat = compute_heatmaps(metric)
    for cat, grid in heat.grids.items():
        write_heatmap_png(grid, heat.bin_cm, PNGS / f"heatmap_{cat}_{stem}.png")
    zones = compute_field_zones(src["tracking_json"], schemes=("mitades", "tercios"), metric=metric)
    for sch in ("mitades", "tercios"):
        write_zones_png(zones, sch, PNGS / f"zonas_{sch}_{stem}.png")
    print(f"  heatmaps={list(heat.grids)}  zonas=mitades,tercios")

[IMG_9933_5m30] métrica (homografía por líneas)…


  heatmaps=['ball', 'robot']  zonas=mitades,tercios
[IMG_9933_8m00] sin insumos -> skip
[IMG_9938_5m00] sin insumos -> skip


## 6 — Cámara superior: velocidad del balón (crudo vs Kalman)
Gráfica que muestra el suavizado/relleno de oclusión del filtro de Kalman (fase_6). CPU.

In [7]:
from src.core.metric_kinematics import compute_kinematics
from src.core.kalman_kinematics import compute_kalman_states

for stem, src in SUPERIOR.items():
    if not (src["tracking_json"] and src["raw"]):
        continue
    metric = compute_metric_positions(src["tracking_json"], video=src["raw"], homography="lines")
    if not any(p.xy_cm is not None for p in metric.posiciones):
        print(f"[{stem}] degradado -> skip gráfica"); continue
    kin = compute_kinematics(metric, with_series=True)   # serie cruda (dif. finitas)
    kres = compute_kalman_states(metric)                 # estados KF (suaviza + rellena oclusión)
    if not kres.por_obj:
        print(f"[{stem}] sin objetos Kalman -> skip"); continue

    # Objeto protagonista: el de mayor distancia recorrida (lista, no dict).
    kobj = max(kres.por_obj, key=lambda o: o.dist_cm)
    kf_frames = [s.frame_index for s in kobj.estados]
    kf_speeds = [s.speed_cms for s in kobj.estados]
    raw_obj = next((o for o in kin.por_obj if o.obj_id == kobj.obj_id and o.serie), None)

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 3.2))
    if raw_obj:
        rf, rv = zip(*raw_obj.serie)
        ax.plot(rf, rv, label="crudo (dif. finitas)", lw=1.0, alpha=0.55)
    ax.plot(kf_frames, kf_speeds, label="Kalman", lw=1.8)
    ax.set_title(f"Velocidad obj#{kobj.obj_id} ({kobj.cls}) — crudo vs Kalman — {stem}")
    ax.set_xlabel("frame"); ax.set_ylabel("cm/s"); ax.legend()
    out = PNGS / f"velocidad_kalman_{stem}.png"
    fig.tight_layout(); fig.savefig(out, dpi=110); plt.close(fig)
    print("  gráfica ->", out, f"(obj#{kobj.obj_id}, {kobj.n_predicted} frames de oclusión rellenados)")
    break  # una gráfica representativa basta para el README

  gráfica -> /mnt/datos/code/ai/futbot/assets/readme/png/velocidad_kalman_IMG_9933_5m30.png (obj#3, 28 frames de oclusión rellenados)


## 7 — Índice de visuales (para el README) + benchmark existente
Lista lo generado en `assets/readme/` y los visuales de benchmark ya versionados.

In [8]:
print("=== assets/readme/gifs ===")
for p in sorted(GIFS.glob("*.gif")):
    print("  ", p.relative_to(PROJECT_ROOT))
print("=== assets/readme/png ===")
for p in sorted(PNGS.glob("*.png")):
    print("  ", p.relative_to(PROJECT_ROOT))
print("=== assets/benchmark (ya versionado) ===")
bench = PROJECT_ROOT / "assets" / "benchmark"
for p in sorted(bench.glob("*.png")) if bench.exists() else []:
    print("  ", p.relative_to(PROJECT_ROOT))

=== assets/readme/gifs ===
   assets/readme/gifs/broadcast_IMG_9933_5m30.gif
   assets/readme/gifs/tracking_video-597_singular_display.gif
   assets/readme/gifs/tracking_video-714_singular_display.gif
=== assets/readme/png ===
   assets/readme/png/broadcast_IMG_9933_5m30.png
   assets/readme/png/heatmap_ball_IMG_9933_5m30.png
   assets/readme/png/heatmap_robot_IMG_9933_5m30.png
   assets/readme/png/velocidad_kalman_IMG_9933_5m30.png
   assets/readme/png/zonas_mitades_IMG_9933_5m30.png
   assets/readme/png/zonas_tercios_IMG_9933_5m30.png
=== assets/benchmark (ya versionado) ===
   assets/benchmark/fase1_detectores.png
   assets/benchmark/fase2_ejes.png
   assets/benchmark/fase2_metricas_debiles.png
   assets/benchmark/fase2_trackers.png
